# Centinel: del dato satelital al modelo y a la API
**Detección de pérdida de bosque en Caquetá y Putumayo**

---
### ⚠️ Preparación: ejecutar ANTES de grabar
Las tres celdas siguientes solo dejan el entorno listo. El video empieza donde dice **AQUÍ EMPIEZA EL VIDEO**.

**Preparación 1.** Subir los archivos. Selecciona a la vez: `centinel_mvp.zip`, `fusion.pt`, `meta.csv` y `Y.npy`.

In [ ]:
from google.colab import files
up = files.upload()

**Preparación 2.** Descomprimir e instalar.

In [ ]:
!unzip -oq centinel_mvp.zip
%cd /content/mvp
!pip install -q rasterio geopandas fastapi uvicorn earthengine-api
!mkdir -p models data/dataset
!mv /content/fusion.pt models/
!mv /content/meta.csv /content/Y.npy data/dataset/
!ls models data/dataset

**Preparación 3.** Conectar Earth Engine. Abre el enlace, acepta y pega el código. **No grabes esta celda**: muestra un código de verificación.

In [ ]:
import ee
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='centinel-tdse')

import os, time, json, requests
import numpy as np, pandas as pd
print("Earth Engine conectado")

---
---
# 🎬 AQUÍ EMPIEZA EL VIDEO

# Paso 1. La pregunta del piloto

**¿Agregar radar (Sentinel-1) a las imágenes ópticas (Sentinel-2) mejora la detección de deforestación anual?**

- Dos configuraciones del mismo modelo (ResNet-18): una ve solo imágenes ópticas, la otra ópticas + radar
- Datos: Sentinel-2 y Sentinel-1 de 2021 a 2023, con etiquetas de pérdida de bosque Hansen/GLAD
- Área: unos 18.000 km² de Caquetá y Putumayo

## Paso 2. Exportación de imágenes desde Earth Engine: 24 zonas, 3 años, 96 tareas
*Plan de exportación. Las tareas ya se ejecutaron antes y terminaron todas.*

In [ ]:
!python 01_export_tiles.py --project centinel-tdse --dry-run | head -n 8

## Paso 3. Los datos del piloto: parches de 1,28 km etiquetados con pérdida de bosque
*Cada parche cubre unos 1,28 km y se marca como positivo si perdió bosque durante el año evaluado.*

In [ ]:
m = pd.read_csv('data/dataset/meta.csv')
y = np.load('data/dataset/Y.npy').ravel()

print(f"Parches de 64x64 píxeles (1,28 km de lado): {len(y):,}")
print(f"Con pérdida de bosque (positivos):          {int(y.sum()):,}  ({y.mean():.1%})")
print(f"Sin pérdida de bosque (negativos):          {int((y == 0).sum()):,}")
print(f"Tiles geográficos:                          {m.tile.nunique()}")
print()
print("Parches por año evaluado:")
print(m.assign(positivo=y).groupby('year').positivo.agg(['count', 'sum'])
        .rename(columns={'count': 'parches', 'sum': 'positivos'}).to_string())

## Paso 4. Entrenamiento: dos configuraciones, tres semillas, en una GPU T4
*Registro del entrenamiento que se hizo antes en Colab. Aquí se muestran la primera y la última época de cada configuración.*

In [ ]:
L = open('results/training_log.txt', encoding='utf-8').read().splitlines()
for i in [0, 1, 15, 16, 30]:     # dispositivo, y primera y última época de cada configuración (semilla 0)
    print(L[i])
print("   ... (6 entrenamientos: 2 configuraciones × 3 semillas)\n")
print("\n".join(L[92:108]))      # medias entre semillas

### Comparación de los dos modelos, con imágenes limpias y con nubes sintéticas
*Cifras medias de tres semillas, medidas en datos que el modelo no vio al entrenar.*

In [ ]:
s = pd.read_csv('results/summary.csv')
nombres = {'precision': 'Precisión', 'recall': 'Recall', 'f1': 'F1',
           'fpr': 'Falsos positivos (FPR)', 'ap': 'Precisión promedio (AP)'}
cond = {'test': 'Imágenes limpias', 'occluded': 'Con nube sintética'}
filas = [(c, mt) for c in ['test', 'occluded'] for mt in nombres]

tab = s.pivot_table(index=['condition', 'metric'], columns='config', values='mean')
tab = tab.loc[filas, ['optical', 'fusion']]
tab.columns = ['Solo óptico', 'Óptico + radar']
tab.index = pd.MultiIndex.from_tuples([(cond[c], nombres[mt]) for c, mt in filas])
print(tab.round(3).to_string())

### Evaluación contra los criterios fijados antes del experimento
*Precisión ≥ 0,70, recall ≥ 0,75 y FPR ≤ 0,20 para el modelo con radar, y que además mejore al modelo solo óptico.*

In [ ]:
v = json.load(open('results/results.json'))['verdict']
f = v['fusion_means']

print("Criterios para adoptar la fusión con radar:")
print(f"  Precisión ≥ 0,70   -> {f['precision']:.3f}  {'CUMPLE' if f['precision'] >= 0.70 else 'NO cumple'}")
print(f"  Recall    ≥ 0,75   -> {f['recall']:.3f}  {'CUMPLE' if f['recall'] >= 0.75 else 'NO cumple'}")
print(f"  FPR       ≤ 0,20   -> {f['fpr']:.3f}  {'CUMPLE' if f['fpr'] <= 0.20 else 'NO cumple'}")
print()
print(f"Mejora de AP con radar, imágenes limpias:   {v['delta_ap_mean']:+.4f}  (despreciable)")
print(f"Mejora de AP con radar, con nube sintética: {v['delta_ap_occluded_mean']:+.4f}  (pequeña, en las 3 semillas)")
print()
print("Resultado:", "criterios cumplidos" if v['passes'] else "el piloto NO cumplió sus criterios")

## Paso 5. Servicio de alertas: arrancar la API y comprobar que responde
*La API carga el modelo entrenado con radar y queda esperando consultas. El estado "ok" confirma que arrancó bien. Tarda unos segundos en cargar el modelo.*

In [ ]:
os.environ['CENTINEL_MODEL'] = 'fusion'
os.environ['CENTINEL_GEE_PROJECT'] = 'centinel-tdse'

!pkill -f uvicorn || true
!cp 04_api.py api.py
!nohup uvicorn api:app --port 8000 > api.log 2>&1 &
time.sleep(20)

print("Estado de la API:")
print(requests.get('http://localhost:8000/health').json())

## Paso 6. Consultar la API: un parche con pérdida de bosque y otro sin pérdida
*La API recibe coordenadas y un año, descarga las imágenes en ese momento y devuelve la probabilidad de pérdida de bosque. Si supera el umbral, lanza una alerta.*

In [ ]:
casos = [
    ("Parche CON pérdida de bosque", m[y == 1].sort_values('loss_frac', ascending=False).iloc[20]),
    ("Parche SIN pérdida de bosque", m[(y == 0) & (m.loss_frac == 0)].iloc[100]),
]

for titulo, p in casos:
    r = requests.get("http://localhost:8000/predict",
                     params=dict(lat=p.lat, lon=p.lon, year=int(p.year)), timeout=120)
    d = r.json()["deforestation"]
    print(titulo)
    print(f"   ubicación: lat {p.lat:.4f}, lon {p.lon:.4f}, año {int(p.year)}")
    print(f"   pérdida real en el parche: {p.loss_frac:.1%}")
    print(f"   probabilidad del modelo:   {d['probability']:.3f}   (umbral {d['threshold']:.3f})")
    print("   ->", "🔴 ALERTA" if d["alert"] else "🟢 sin alerta")
    print()

## Paso 7. Comprobar el modelo en zonas nuevas: 20 parches con pérdida y 20 sin pérdida
*Los parches vienen de zonas que el modelo no vio al entrenar. Se consultan a la API y se cuentan aciertos y falsas alarmas.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def split_by_group(groups, seed):      # misma división que usó el entrenamiento
    idx = np.arange(len(groups))
    tr, rest = next(GroupShuffleSplit(1, test_size=0.30, random_state=seed).split(idx, groups=groups))
    va_rel, te_rel = next(GroupShuffleSplit(1, test_size=0.50, random_state=seed)
                          .split(rest, groups=groups[rest]))
    return tr, rest[va_rel], rest[te_rel]

_, _, te = split_by_group(m.tile.values, seed=0)
print(f"Parches de prueba: {len(te)} (deben ser 1873), con pérdida: {int(y[te].sum())} (deben ser 268)")

rng = np.random.default_rng(0)
pos_idx = rng.choice(te[y[te] == 1], 20, replace=False)
neg_idx = rng.choice(te[y[te] == 0], 20, replace=False)

filas = []
for i in list(pos_idx) + list(neg_idx):
    p = m.iloc[i]
    r = requests.get("http://localhost:8000/predict",
                     params=dict(lat=p.lat, lon=p.lon, year=int(p.year)), timeout=120)
    if r.status_code != 200:
        continue
    d = r.json()["deforestation"]
    filas.append(dict(real=int(y[i]), prob=d['probability'], alerta=bool(d['alert']),
                      loss_frac=float(p.loss_frac)))

res = pd.DataFrame(filas)
tp = int(((res.real == 1) & res.alerta).sum());  fn = int(((res.real == 1) & ~res.alerta).sum())
fp = int(((res.real == 0) & res.alerta).sum());  tn = int(((res.real == 0) & ~res.alerta).sum())

print(f"\nConsultas respondidas: {len(res)} de 40")
print(pd.DataFrame([[tp, fn], [fp, tn]],
      index=['Había pérdida', 'No había pérdida'],
      columns=['Alerta', 'Sin alerta']))
print(f"\nDetectó {tp} de {tp + fn} parches con pérdida  (recall {tp / max(tp + fn, 1):.2f})")
print(f"Falsas alarmas: {fp} de {fp + tn} parches sin pérdida  (FPR {fp / max(fp + tn, 1):.2f})")

## Paso 8. Qué se comprobó y qué no

**Se comprobó**
- Se descargaron y procesaron imágenes reales de Sentinel-1 y Sentinel-2 desde Earth Engine
- Se entrenaron y compararon dos configuraciones del modelo, con tres semillas cada una
- La API se ejecutó como servidor en Colab, consultó Earth Engine en vivo y devolvió alertas

**No se comprobó**
- La prueba de la API es de funcionamiento, con muestras pequeñas: no reemplaza la evaluación del entrenamiento
- Las etiquetas Hansen/GLAD miden pérdida de bosque, no su causa ni su legalidad
- No se midieron latencia ni rendimiento
- La API **todavía no está desplegada en la nube**